# 09 County archetypes: K means vs Gaussian: +,- policy
>by Khalid Jackson & Tahreem Karim

Ntbk 03 (Khalid) found **three stable K means archetypes** on the 11 CERF features (electricity capped at the
99th percentile). This will extend Part B by:

1. Refitting that K means model & regenerating figs B2/B3 so labels/colors match on report/dashboard.
2. **Gaussian mixture models** (probabilistic, soft membership): K = 2-10 × full / diagonal covariance,
   judged by BIC, silhouette of hard labels and smallest component (Fig. B4, Table B1).
3. **Membership uncertainty**: are borderline counties more or less likely to host a data center?
4. **External validation** (Table B2): share of each archetype w/ existing facility & chosen by PNNL.
5. **Policy-augmented clustering**: add nine state policy features. If new clusters line up w/
   state lines (ARI / NMI against state labels) basically just says "which state".

**Outputs** `data/cerf_siting/output/stage5_unsupervised/`: `clusters.csv`, `gmm_sweep.csv`, `gmm_reg_covar.csv`, `table_b1.csv`,
`table_b2.csv`, `uncertainty_vs_prevalence.csv`, `policy_clustering.csv`, `kmeans_profiles_*.csv`, `pca_coords.csv`,
`pca_variance.csv` (figures drawn in ntbk 10).

In [ ]:
import warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

from pipeline_config import CLUSTER_FEATURES, ELECTRICITY, GROUP, POLICY, STAGE5, STAGE6, TARGET, load_model_table

warnings.filterwarnings("ignore")
NAMES = ["Tax-light, pricey land", "Hot or dry, grid-remote", "Grid-close, cheap land"]   # cluster 1 was "Dry, grid-remote"
# a third of its counties are in the humid Southeast so name now says that they share "cooling must be mechanical" quality
sil = lambda X, lab: silhouette_score(X, lab, sample_size=2000, random_state=42)   # same convention as notebook 03

d = load_model_table()
Xraw = d[CLUSTER_FEATURES].copy()
for c in ELECTRICITY:
    Xraw[c] = Xraw[c].clip(upper=Xraw[c].quantile(0.99))

X = StandardScaler().fit_transform(Xraw)
print(f"{len(d):,} counties × {X.shape[1]} features; scaled range {X.min():.2f} to {X.max():.2f}")

3,099 counties × 11 features; scaled range -4.03 to 52.19


## K-means w/ stable names

K = 3 (see ntbk 03). Clusters are renumbered by their defining trait so numbering never depends on the random start.
- 0 = lowest personal property tax
- 1 = highest mechanical cooling 
- 2 = the rest

In [ ]:
km = KMeans(3, random_state=42, n_init=10).fit(X)
g = d.assign(_l=km.labels_).groupby("_l")

c0 = g.personal_property_tax_rate_mean.mean().idxmin()
c1 = g.mechanical_cooling_fraction_mean.mean().drop(c0).idxmax()
c2 = ({0, 1, 2} - {c0, c1}).pop()

relabel = {c0: 0, c1: 1, c2: 2}

d["kmeans"] = pd.Series(km.labels_).map(relabel).values
dist = np.sort(km.transform(X), axis=1)
d["kmeans_fit_ratio"] = dist[:, 0] / dist[:, 1]          # 0 = at a center, 1 = equidistant from two centers

km_sil = sil(X, d.kmeans)
print("sizes", d.kmeans.value_counts().sort_index().tolist(), f"| silhouette {km_sil:.3f} | inertia {km.inertia_:,.2f}")

sizes [739, 651, 1709] | silhouette 0.261 | inertia 23,891.56


> Three archetypes sized 739 / 651 / 1,709 counties w/ small silhouette (0.261) meaning a "real" but not sharply separated structure.

## Gaussian mixture sweep (Figure B4)

`reg_covar = 1e-3` keeps covariances invertible. *Personal property tax takes only two values* (0 or 0.0125)
& cooling fractions pile at 0 & 1 (some components would otherwise become "flat").

In [ ]:
sweep = []
for cov in ["full", "diag"]:
    for k in range(2, 11):
        gm = GaussianMixture(k, covariance_type=cov, reg_covar=1e-3, n_init=5, random_state=42).fit(X)
        lab = gm.predict(X)
        sweep.append({"covariance": cov, "K": k, "bic": gm.bic(X), "aic": gm.aic(X),
                      "silhouette": sil(X, lab) if len(set(lab)) > 1 else np.nan,
                      "smallest": np.bincount(lab, minlength=k).min(), "ari_vs_kmeans": adjusted_rand_score(d.kmeans, lab)})

sweep = pd.DataFrame(sweep)
sweep.to_csv(STAGE5 / "gmm_sweep.csv", index=False)
sweep.round(3)

,covariance,K,bic,aic,silhouette,smallest,ari_vs_kmeans
0,full,2,36251.784,35315.765,0.241,969,0.339
1,full,3,14813.004,13405.955,0.210,681,0.717
2,full,4,1997.806,119.729,0.212,151,0.656
3,full,5,-966.923,-3316.030,0.201,149,0.615
4,full,6,-4233.007,-7053.143,0.195,29,0.618
5,full,7,-3471.329,-6762.493,0.233,1,0.633
6,full,8,-4182.058,-7944.252,0.105,101,0.362
7,full,9,-4765.054,-8998.277,0.056,102,0.315
8,full,10,-7684.432,-12388.684,0.122,29,0.373
9,diag,2,46009.554,45737.806,0.246,949,0.349


>BIC keeps improving as more components get added. However, at higher K some components shrink to single county (overfitting).

### Choosing mixture

BIC falls as components are added (features not Gaussian, binary tax rates & cooling
fractions at 0 or 1 reward extra components), so BIC minimum is not a usable choice. We keep **K = 3,
full covariance** for a comparison with K means & report BIC preferred model too.

In [ ]:
gm3 = GaussianMixture(3, covariance_type="full", reg_covar=1e-3, n_init=5, random_state=42).fit(X)
P = gm3.predict_proba(X)
d["gmm"] = P.argmax(1)

# align GMM component numbers w/ K means archetypes they overlap most
overlap = pd.crosstab(d.gmm, d.kmeans)
gmap = {}
for comp in overlap.max(axis=1).sort_values(ascending=False).index:
    best = [a for a in overlap.loc[comp].sort_values(ascending=False).index if a not in gmap.values()][0]
    gmap[comp] = best

d["gmm"] = d.gmm.map(gmap)
P = P[:, [k for k, _ in sorted(gmap.items(), key=lambda kv: kv[1])]]
d["gmm_entropy"] = -(P * np.log(np.clip(P, 1e-12, 1))).sum(1) / np.log(3)
best_bic = sweep.loc[sweep.bic.idxmin()]

# How confident is mixture? Near binary features (personal property tax is 0 or 0.0125 by state, cooling
# fractions grow either at 0 & 1) making components very sharp, so memberships are close to 0 or 1.
reg = []
for rc in [1e-3, 1e-2, 1e-1, 1.0]:
    g = GaussianMixture(3, covariance_type="full", reg_covar=rc, n_init=5, random_state=42).fit(X)
    pp = g.predict_proba(X)
    ent = -(pp * np.log(np.clip(pp, 1e-12, 1))).sum(1) / np.log(3)
    reg.append({"reg_covar": rc, "share_entropy_below_0.1": (ent < 0.1).mean(), "median_entropy": np.median(ent),
                "silhouette": sil(X, pp.argmax(1)), "ari_vs_kmeans": adjusted_rand_score(d.kmeans, pp.argmax(1))})

pd.DataFrame(reg).to_csv(STAGE5 / "gmm_reg_covar.csv", index=False)
print(pd.DataFrame(reg).round(3))
print(f"GMM K=3 full: sizes {d.gmm.value_counts().sort_index().tolist()}, silhouette {sil(X, d.gmm):.3f}, "
      f"ARI vs K-means {adjusted_rand_score(d.kmeans, d.gmm):.3f}")
print(f"BIC minimum: K={best_bic.K} {best_bic.covariance}, smallest component {best_bic.smallest} counties")

table_b1 = pd.DataFrame([
    {"method": "K-means (capped CERF)", "K": 3, "silhouette": km_sil, "bic": np.nan,
     "smallest": d.kmeans.value_counts().min(), "ari_vs_kmeans": 1.0},
    {"method": "Gaussian mixture, full covariance", "K": 3, "silhouette": sil(X, d.gmm), "bic": gm3.bic(X),
     "smallest": d.gmm.value_counts().min(), "ari_vs_kmeans": adjusted_rand_score(d.kmeans, d.gmm)},
    {"method": f"Gaussian mixture, BIC minimum ({best_bic.covariance})", "K": int(best_bic.K),
     "silhouette": best_bic.silhouette, "bic": best_bic.bic, "smallest": int(best_bic.smallest),
     "ari_vs_kmeans": best_bic.ari_vs_kmeans}])

table_b1.to_csv(STAGE5 / "table_b1.csv", index=False)
table_b1.round(3)

   reg_covar  share_entropy_below_0.1  median_entropy  silhouette  \
0      0.001                    0.994           0.000       0.210   
1      0.010                    0.984           0.000       0.226   
2      0.100                    0.959           0.001       0.242   
3      1.000                    0.967           0.009       0.518   

   ari_vs_kmeans  
0          0.717  
1          0.783  
2          0.868  
3          0.014  
GMM K=3 full: sizes [681, 966, 1452], silhouette 0.210, ARI vs K-means 0.717
BIC minimum: K=10 full, smallest component 29 counties


,method,K,silhouette,bic,smallest,ari_vs_kmeans
0,K-means (capped CERF),3,0.261,NaN,651,1.000
1,"Gaussian mixture, full covariance",3,0.210,14813.004,681,0.717
2,"Gaussian mixture, BIC minimum (full)",10,0.122,-7684.432,29,0.373


In [ ]:
# Table B1 as reported: silhouette on all 3,099 counties (not the 2,000-county sample above) and Davies-Bouldin (lower is better)
from sklearn.metrics import davies_bouldin_score

gm_bic = GaussianMixture(int(best_bic.K), covariance_type=best_bic.covariance, reg_covar=1e-3, n_init=5, random_state=42).fit(X)
labels = {"K-means (capped CERF)": d.kmeans.values, "Gaussian mixture, full covariance": d.gmm.values,
          f"Gaussian mixture, BIC minimum ({best_bic.covariance})": gm_bic.predict(X)}
pd.DataFrame({"silhouette_all_counties": {m: silhouette_score(X, lab) for m, lab in labels.items()},
              "davies_bouldin": {m: davies_bouldin_score(X, lab) for m, lab in labels.items()}}).round(3)

,silhouette_all_counties,davies_bouldin
K-means (capped CERF),0.263,1.499
"Gaussian mixture, full covariance",0.216,1.721
"Gaussian mixture, BIC minimum (full)",0.126,2.230


>Chosen setup (`reg_covar=1e-3`) is stable: 96-99% of counties stay “clearly” assigned (entropy < 0.1) from `reg_covar` 1e-3 to 0.1. GMM at K=3 recovers groups that largely agree with K-means (ARI 0.72). BIC-preferred alternative (K=10) fits better numerically but produces an almost empty 29 county cluster w/ a weaker silhouette. Not a better real world grouping, but more flexible.

## Membership uncertainty & data center prevalence
Checks whether how confidently a county belongs to an archetype (vs. being ambiguously btwn. two) says anything about having a data center.

In [ ]:
d["entropy_band"] = pd.cut(d.gmm_entropy, [-0.01, 0.1, 0.5, 1.01], labels=["Clear (< 0.1)", "Moderate", "Borderline (> 0.5)"])
d["fit_band"] = pd.qcut(d.kmeans_fit_ratio, 3, labels=["Near a center", "Middle", "Between two"])

unc = pd.concat([
    d.groupby("entropy_band", observed=True).agg(counties=(TARGET, "size"), share_with_dc=(TARGET, "mean")).assign(measure="GMM entropy"),
    d.groupby("fit_band", observed=True).agg(counties=(TARGET, "size"), share_with_dc=(TARGET, "mean")).assign(measure="K-means distance ratio")])

rho_e = spearmanr(d.gmm_entropy, d[TARGET]).statistic
rho_k = spearmanr(d.kmeans_fit_ratio, d[TARGET]).statistic

print(f"Spearman with has_data_center: GMM entropy {rho_e:+.3f}, K-means distance ratio {rho_k:+.3f}")
unc.to_csv(STAGE5 / "uncertainty_vs_prevalence.csv")
unc.round(3)

Spearman with has_data_center: GMM entropy -0.013, K-means distance ratio +0.086


,counties,share_with_dc,measure
Clear (< 0.1),3081,0.078,GMM entropy
Moderate,13,0.231,GMM entropy
Borderline (> 0.5),5,0.400,GMM entropy
Near a center,1033,0.057,K-means distance ratio
Middle,1033,0.066,K-means distance ratio
Between two,1033,0.114,K-means distance ratio


>Correlation is pretty much flat (Spearman ≈ 0). The GMM's moderate and borderline bands show higher data center rates (23% and 40%), but they hold only 13 and 5 counties, so they can't carry a conclusion. The K-means distance ratio splits all counties into thirds: the most borderline third has about twice the data center rate of the third nearest a center (11.4% vs 5.7%). Basically borderline counties are "somewhat more likely" to already have a data center.

## Table B2 Data center prevalence by archetype (external validation)
Checks whether the archetypes from infrastructure data line up w/ actual data center presence & PNNL’s own projected sites, by cluster.

In [ ]:
pn = pd.read_csv(STAGE6 / "pnnl_county.csv", dtype={"county_fips": str}).set_index("county_fips")
d = d.join(pn, on="county_fips")
rows = []

for method in ["kmeans", "gmm"]:
    for k in range(3):
        s = d[d[method] == k]
        mod = s[s.pnnl_modeled]
        rows.append({"method": "K-means" if method == "kmeans" else "GMM (K = 3)", "cluster": k, "name": NAMES[k],
                     "counties": len(s), "share_with_dc": s[TARGET].mean(), "facilities": int(s.atlas_record_count.sum()),
                     "share_pnnl_ref": mod.pnnl_ref.mean(), "pnnl_sites": int(mod.n_projected_ref.sum())})

table_b2 = pd.DataFrame(rows)
table_b2.to_csv(STAGE5 / "table_b2.csv", index=False)

print(f"overall: {d[TARGET].mean():.1%} with a data center; "
      f"{d[d.pnnl_modeled].pnnl_ref.mean():.1%} of PNNL-modeled counties chosen in the reference scenario")
table_b2.round(3)

overall: 7.9% with a data center; 2.8% of PNNL-modeled counties chosen in the reference scenario


,method,cluster,name,counties,share_with_dc,facilities,share_pnnl_ref,pnnl_sites
0,K-means,0,"Tax-light, pricey land",739,0.104,342,0.029,72
1,K-means,1,"Hot or dry, grid-remote",651,0.061,294,0.012,40
2,K-means,2,"Grid-close, cheap land",1709,0.075,836,0.033,216
3,GMM (K = 3),0,"Tax-light, pricey land",681,0.107,321,0.031,72
4,GMM (K = 3),1,"Hot or dry, grid-remote",966,0.066,385,0.020,59
5,GMM (K = 3),2,"Grid-close, cheap land",1452,0.074,766,0.031,197


>Tax light, pricey land counties have highest real data center rate (10.4-10.7%). Hot or dry, grid-remote lowest (6.1-6.6%), & PNNL agrees that archetype is least likely (1.2%), but its picks favor grid-close, cheap land counties (3.3%, 216 of its 328 sites), in line with CERF-DC's cost logic. The archetypes are not arbitrary, but they separate siting outcomes only modestly (normalized mutual information with the label 0.002, see policy clustering below).

## Policy augmented clustering

The nine state policy features are standardized and appended to the 11 CERF features. Agreement with
plain state labels (48 states) and Census divisions is the test: high values mean the "archetypes" are
mostly state lines.

In [ ]:
Xp = np.hstack([X, StandardScaler().fit_transform(d[POLICY])])
res = []

for name, Z in [("CERF only", X), ("CERF + policy", Xp)]:
    for method in ["K-means", "GMM"]:
        
        for k in ([3] if method == "GMM" else [3, 4, 5]):
            m = (KMeans(k, random_state=42, n_init=10) if method == "K-means"
                 else GaussianMixture(k, covariance_type="full", reg_covar=1e-3, n_init=5, random_state=42)).fit(Z)
            lab = m.predict(Z)
            res.append({"features": name, "method": method, "K": k, "silhouette": sil(Z, lab),
                        "ari_vs_cerf_kmeans": adjusted_rand_score(d.kmeans, lab),
                        "ari_vs_state": adjusted_rand_score(d[GROUP], lab), "nmi_vs_state": normalized_mutual_info_score(d[GROUP], lab),
                        "nmi_vs_division": normalized_mutual_info_score(d.division, lab),
                        "share_states_split": (d.assign(l=lab).groupby(GROUP).l.nunique() > 1).mean(),
                        "nmi_vs_label": normalized_mutual_info_score(d[TARGET], lab),
                        "dc_share_by_cluster": " / ".join(f"{v:.1%}" for v in d.groupby(lab)[TARGET].mean()),
                        "smallest": np.bincount(lab).min()})

policy_clustering = pd.DataFrame(res)
policy_clustering.to_csv(STAGE5 / "policy_clustering.csv", index=False)
policy_clustering.round(3)

,features,method,K,silhouette,ari_vs_cerf_kmeans,ari_vs_state,nmi_vs_state,nmi_vs_division,share_states_split,nmi_vs_label,dc_share_by_cluster,smallest
0,CERF only,K-means,3,0.261,1.000,0.054,0.307,0.231,0.500,0.002,10.4% / 6.1% / 7.5%,651
1,CERF only,K-means,4,0.261,1.000,0.054,0.307,0.231,0.521,0.003,6.1% / 7.5% / 10.3% / 100.0%,1
2,CERF only,K-means,5,0.243,0.725,0.072,0.331,0.225,0.812,0.006,6.1% / 10.3% / 6.0% / 11.6% / 100.0%,1
3,CERF only,GMM,3,0.210,0.717,0.057,0.283,0.231,0.792,0.002,6.6% / 10.7% / 7.4%,681
4,CERF + policy,K-means,3,0.156,0.303,0.063,0.318,0.163,0.521,0.002,6.1% / 7.3% / 10.1%,623
5,CERF + policy,K-means,4,0.151,0.340,0.123,0.440,0.215,0.542,0.001,8.8% / 8.3% / 6.6% / 6.7%,516
6,CERF + policy,K-means,5,0.170,0.437,0.148,0.493,0.320,0.542,0.004,14.4% / 9.2% / 7.6% / 4.8% / 6.6%,194
7,CERF + policy,GMM,3,0.122,0.238,0.059,0.291,0.157,0.771,0.002,6.0% / 8.3% / 9.7%,712


In [8]:
d[["county_fips", "kmeans", "kmeans_fit_ratio", "gmm", "gmm_entropy"]].assign(
    cluster_name=d.kmeans.map(dict(enumerate(NAMES)))).to_csv(STAGE5 / "clusters.csv", index=False)

>Adding policy barely changes how clusters trace state lines (low 0.05-0.15). But it does meaningfully reshuffle groups themselves (agreement w/ original CERF only clusters drops 1.00 to 0.30 at K=3, silhouette weakens). Policy pulls the structure in an actual different direction but not as tightly separated.

## Data for Figures B2-B5 (one fitted model)

Profiles (SD from the all-county mean), PCA coordinates and the GMM sweep are saved for notebook 10.

In [ ]:
prof = (d.groupby("kmeans")[CLUSTER_FEATURES].mean() - d[CLUSTER_FEATURES].mean()) / d[CLUSTER_FEATURES].std()
pct = (d.groupby("kmeans")[CLUSTER_FEATURES].mean() / d[CLUSTER_FEATURES].mean() - 1)
prof.T.to_csv(STAGE5 / "kmeans_profiles_sd.csv")
pct.T.to_csv(STAGE5 / "kmeans_profiles_pct.csv")

pca = PCA(2).fit(X)
Z2 = pca.transform(X)
ev = pca.explained_variance_ratio_

pd.DataFrame({"county_fips": d.county_fips, "pc1": Z2[:, 0], "pc2": Z2[:, 1], "kmeans": d.kmeans,
              TARGET: d[TARGET]}).to_csv(STAGE5 / "pca_coords.csv", index=False)

pd.Series({"pc1": ev[0], "pc2": ev[1]}).to_csv(STAGE5 / "pca_variance.csv")

print(f"sizes {d.kmeans.value_counts().sort_index().tolist()} | PC1 {ev[0]:.1%} + PC2 {ev[1]:.1%} = {ev.sum():.1%}")
prof.T.round(2)

sizes [739, 651, 1709] | PC1 22.6% + PC2 15.4% = 38.0%


kmeans,0,1,2
commercial_electricity_rates_dollar_per_kwh_mean,-0.06,0.16,-0.03
industrial_electricity_rates_dollar_per_kwh_mean,-0.02,-0.02,0.01
composite_siting_suitability_mean,-0.13,-0.22,0.14
distance_to_substation_mean,-0.06,0.53,-0.17
distance_to_substation_min,-0.02,0.17,-0.06
land_value_dollar_per_sqft_mean,0.12,-0.06,-0.03
mechanical_cooling_fraction_mean,-0.52,1.85,-0.48
personal_property_tax_rate_mean,-1.74,0.48,0.57
real_property_tax_rate_mean,0.94,-0.06,-0.38
sales_tax_rate_mean,-0.06,0.12,-0.02


>The 2D PCA view only captures 38% of total variance (22.6% + 15.4%), so it’s a useful window, not the full thing. The per cluster profile confirms archetype names make sense. Cluster 1 (“*Hot or dry*”) is extreme on mechanical cooling (+1.85 SD) and far below average on water cooling (-1.85 SD). Cluster 0 is far below average on personal property tax (-1.74 SD) aka “tax-light."